# 07 機械学習モデル(Tier0-2、walk-forward OOS)

`irp.models` は共通の fit/predict IF を持つ:**Tier0 ベースライン**(zero/mean/persistence)がバー、**Tier1 linear**(Ridge)・**Tier2 tree**(RandomForest)は *同じ OOS walk* でそれを超えて初めて価値がある。features → labels → `make_design` → **fold ごとに学習**(`walk_forward_predict`)→ OOS 予測 → IC → weights → 既存エンジンでバックテスト → **ベースラインと並べる**。

In [ ]:
import numpy as np
import pandas as pd

# Synthetic panel where forward returns are PARTLY predictable: a mild per-asset
# momentum (AR(1)) makes trailing features genuinely informative, plus noise so
# nothing is trivially learnable. No network. (Real data: irp.data.price_panel.)
rng = np.random.default_rng(3)
idx = pd.bdate_range('2015-01-01', periods=1600)
names = [f'A{i}' for i in range(8)]
n, k = len(idx), len(names)
eps = rng.normal(0, 0.012, (n, k))
r = eps.copy()
for t in range(1, n):
    r[t] += 0.06 * r[t - 1]              # mild momentum -> features have signal
rets = pd.DataFrame(r, index=idx, columns=names)
prices = 100 * np.exp(rets.cumsum())
prices.iloc[-1].round(1)

## 特徴量 + ラベル → 設計行列((date, asset) スタック、前埋めなし)

In [ ]:
from irp import features as F, models as MD, signals as S, backtest as B
from irp.labels import forward_return

H = 5  # label horizon (bars)
feats = {
    'mom_120_20': F.momentum(prices, lookback=120, skip=20),
    'z_20': F.rolling_zscore(prices, 20),
    'rvol_20': F.realized_volatility(prices, 20),
    'ma_ratio_50': F.ma_ratio(prices, 50),
}
label = forward_return(prices, horizon=H)
X, y, dropped = MD.make_design(feats, label, return_dropped=True)
print(f'design matrix X={X.shape}, dropped {dropped} incomplete rows (no fill)')
X.head(3)

## walk-forward fold(purge+embargo)と OOS 予測

In [ ]:
folds = B.walk_forward(prices.index, train=252, test=63, horizon=H, embargo=3)
leak_free = all(B.is_leakage_free(f, prices.index, horizon=H) for f in folds)
print(f'{len(folds)} folds, all leakage-free: {leak_free}')

model_factories = {
    'persist[mom]': lambda: MD.PersistenceModel('mom_120_20'),  # Tier0
    'mean':         lambda: MD.MeanModel(),                     # Tier0
    'ridge':        lambda: MD.ridge(alpha=1.0),                # Tier1
    'random_forest':lambda: MD.random_forest(n_estimators=150, max_depth=3),  # Tier2
}
oos = {name: MD.walk_forward_predict(f, X, y, folds) for name, f in model_factories.items()}
print('OOS predictions per model:', {k: len(v) for k, v in oos.items()})

## 情報係数(IC)= 予測と実現フォワードリターンの横断順位相関(OOS)

ML-for-finance の標準指標。日次で横断 Spearman を取り平均。Tier0 を超えるかが鍵。

In [ ]:
def mean_ic(pred, realized):
    df = pd.DataFrame({'p': pred, 'r': realized.loc[pred.index]}).dropna()
    def _c(g):  # undefined for <3 names or a constant prediction (e.g. mean model)
        if len(g) < 3 or g['p'].nunique() < 2 or g['r'].nunique() < 2:
            return np.nan
        return g['p'].corr(g['r'], method='spearman')
    return float(df.groupby(level='date').apply(_c).mean())

ic = pd.Series({name: mean_ic(p, y) for name, p in oos.items()}, name='mean_IC')
ic.round(4).sort_values(ascending=False)

## 予測 → weights → バックテスト → ベースライン比較

各モデルの OOS 予測を横断標準化して dollar-neutral の L/S weight にし、**ラベル horizon (H 日)に合わせてリバランス**(日次だとターンオーバーが病的でコストが全てを食う)。同じエンジン(lag・コスト)でバックテストし、等加重バイ&ホールドと並べる。

In [ ]:
from irp.features.price import cross_sectional_zscore
from irp.utils.config import load_config

cost = B.CostModel.from_config(load_config('backtest_config'))
oos_dates = pd.DatetimeIndex(
    sorted(set().union(*[p.index.get_level_values('date') for p in oos.values()]))
)
results = {}
for name, pred in oos.items():
    panel = MD.predictions_to_panel(pred).reindex(oos_dates)
    score = cross_sectional_zscore(panel)
    w_h = S.long_short_quantile(score.iloc[::H], quantile=0.3)  # rebalance every H bars
    w = w_h.reindex(oos_dates, method='ffill')                  # hold between rebalances
    results[name] = B.run_backtest(w, rets, cost_model=cost, lag=1)
results['equal_weight'] = B.buy_and_hold(rets.loc[oos_dates])
B.compare(results, periods=252).round(4)

**読み方**: Tier1/2 の Sharpe・IC が Tier0(persist/mean)と等加重を **OOS で** 超えて初めて複雑さが正当化される。超えなければ「複雑モデルは効かない／コストに負ける」を提示する(それも結果)。IC が薄く正でもターンオーバー次第でコスト後は負ける——という教訓が要点。

**正直な注記**: 合成データ・少資産・単純コスト・税/制約未考慮・ハイパラ未調整・横断 IC は少資産でノイジー。*方法論* の実演であり投資結果ではない。次は税(NISA after-tax)→ portfolio → dashboard → HTML。